# UR10e Debugging helps 

First I need to see the Robot starting position and the rendering camera

In [1]:
import jax
import jax.numpy as jnp
import mediapy as media
from mujoco_playground._src import registry

# -------------------------
# Config
# -------------------------
env_name = "UR10PickCube"
seed = 0
episode_length = 250
render_every = 1
camera_name = "side_130"

# -------------------------
# Load env
# -------------------------
env = registry.load(env_name)
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

rng = jax.random.PRNGKey(seed)
rng, reset_rng = jax.random.split(rng)
state = jit_reset(reset_rng)

# -------------------------
# Infer action dimension
# -------------------------
if hasattr(env, "action_size"):
    act_dim = int(env.action_size)
elif hasattr(env, "action_dim"):
    act_dim = int(env.action_dim)
else:
    raise RuntimeError("Cannot infer action dimension")

# -------------------------
# Zero-action policy (pure kinematics view)
# -------------------------
def zero_policy(_obs, _rng):
    return jnp.zeros((act_dim,), dtype=jnp.float32)

# -------------------------
# Rollout
# -------------------------
rollout = [state]
for _ in range(episode_length):
    rng, a_rng = jax.random.split(rng)
    action = zero_policy(state.obs, a_rng)
    state = jit_step(state, action)
    rollout.append(state)

trajectory = rollout[::render_every]

# -------------------------
# Render inline (NO saving)
# -------------------------
frames = env.render(trajectory, camera=camera_name)
fps = float(1.0 / env.dt) / float(render_every)

media.show_video(frames, fps=fps)


Failed to import warp: No module named 'warp'
Failed to import mujoco.mjx.third_party.mujoco_warp as mujoco_warp: No module named 'warp'
XML PATH: /Users/matthiasweiss/Desktop/ZHAW/MSE/3_VT1_Model Based RL/My_Mujoco/my_mujoco_playground/mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_single_cube_position.xml


NotImplementedError: (mjtGeom.mjGEOM_CYLINDER, mjtGeom.mjGEOM_BOX) collisions not implemented.